# Depth Anything V2: Dataset Processing and Training Notebook

This notebook preprocesses a custom depth dataset (including HDF5), builds reproducible train/val splits, and trains Depth Anything V2 with `accelerate`.

Recommended dataset schema per sample (HDF5):
- `rgb_scene`: uint8, shape `[H, W, 3]` (RGB)
- `depth_planar_m`: float32, shape `[H, W]` (meters)
- Optional: `segmentation_rgb`: uint8, shape `[H, W, 3]`
- Optional: `segmentation_instance_id`: uint32, shape `[H, W]`

## 1. Environment Setup and Repository Dependencies

In [ ]:
# If needed, uncomment to install dependencies.
# !pip install -q torch torchvision opencv-python albumentations h5py accelerate transformers tqdm matplotlib pyyaml
# !git clone https://github.com/DepthAnything/Depth-Anything-V2
# !wget -O depth_anything_v2_vits.pth https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth?download=true

import os
import json
import csv
import random
from pathlib import Path

import cv2
import h5py
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A

from accelerate import Accelerator
from accelerate.utils import set_seed
from accelerate import DistributedDataParallelKwargs

import transformers

# Update this path to your local Depth-Anything-V2 repo if needed.
import sys
depth_anything_repo = Path('./Depth-Anything-V2/metric_depth').resolve()
if depth_anything_repo.exists():
    sys.path.append(str(depth_anything_repo))

from depth_anything_v2.dpt import DepthAnythingV2
from util.loss import SiLogLoss
from dataset.transform import Resize, NormalizeImage, PrepareForNet, Crop

print('Imports loaded.')

## 2. Project Paths and Dataset Configuration

In [ ]:
CONFIG = {
    'raw_dataset_root': r'E:/Programs/AirSim/Cosys-AirSim/PythonClient/YOLO-3D/dataset',
    'processed_root': r'E:/Programs/AirSim/Cosys-AirSim/PythonClient/YOLO-3D/processed_dataset',
    'checkpoint_root': r'E:/Programs/AirSim/Cosys-AirSim/PythonClient/YOLO-3D/checkpoints_depth_anything',
    'output_root': r'E:/Programs/AirSim/Cosys-AirSim/PythonClient/YOLO-3D/outputs_depth_anything',
    'train_manifest': 'train_manifest.csv',
    'val_manifest': 'val_manifest.csv',
    'cache_manifest': 'cache_manifest.csv',
    'seed': 42,
    'img_size': (518, 518),
    'max_depth_m': 80.0,
    'min_depth_m': 0.001,
    'batch_size_train': 8,
    'batch_size_val': 1,
    'num_workers': 4,
    'pin_memory': True,
    'persistent_workers': True,
    'num_epochs': 10,
    'learning_rate': 5e-6,
    'weight_decay': 1e-2,
    'warmup_epochs': 0.5,
    'scheduler_rate': 1.0,
    'mixed_precision': 'fp16',
    'model_encoder': 'vits',
    'pretrained_weights_path': r'E:/Programs/AirSim/Cosys-AirSim/PythonClient/YOLO-3D/depth_anything_v2_vits.pth',
    'split_ratio': 0.9,
    'split_file': 'split_index.json',
    'depth_clip_enabled': True,
    'depth_normalize_enabled': False,
    'segmentation_filter_enabled': False,
    'segmentation_min_foreground_ratio': 0.001,
}

for k in ['processed_root', 'checkpoint_root', 'output_root']:
    Path(CONFIG[k]).mkdir(parents=True, exist_ok=True)

set_seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
print(json.dumps(CONFIG, indent=2))

## 3. Raw Dataset Indexing and Split Generation

In [ ]:
def discover_h5_samples(raw_root):
    raw_root = Path(raw_root)
    candidates = sorted(raw_root.rglob('*.h5'))
    samples = []
    for p in candidates:
        samples.append({'h5_path': str(p.resolve())})
    return samples

def deterministic_split(samples, split_ratio=0.9, seed=42):
    idxs = list(range(len(samples)))
    rng = random.Random(seed)
    rng.shuffle(idxs)
    n_train = int(len(idxs) * split_ratio)
    train_idx = set(idxs[:n_train])
    train_samples, val_samples = [], []
    for i, s in enumerate(samples):
        (train_samples if i in train_idx else val_samples).append(s)
    return train_samples, val_samples

def save_manifest_csv(samples, out_csv):
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with open(out_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['h5_path'])
        writer.writeheader()
        for row in samples:
            writer.writerow({'h5_path': row['h5_path']})

all_samples = discover_h5_samples(CONFIG['raw_dataset_root'])
train_samples, val_samples = deterministic_split(
    all_samples, split_ratio=CONFIG['split_ratio'], seed=CONFIG['seed']
 )

split_json_path = Path(CONFIG['processed_root']) / CONFIG['split_file']
split_json_path.write_text(
    json.dumps({
        'seed': CONFIG['seed'],
        'split_ratio': CONFIG['split_ratio'],
        'num_total': len(all_samples),
        'num_train': len(train_samples),
        'num_val': len(val_samples),
    }, indent=2),
    encoding='utf-8'
 )

train_manifest_path = Path(CONFIG['processed_root']) / CONFIG['train_manifest']
val_manifest_path = Path(CONFIG['processed_root']) / CONFIG['val_manifest']
save_manifest_csv(train_samples, train_manifest_path)
save_manifest_csv(val_samples, val_manifest_path)

print(f'Total samples: {len(all_samples)}')
print(f'Train samples: {len(train_samples)}')
print(f'Val samples:   {len(val_samples)}')
print(f'Train manifest: {train_manifest_path}')
print(f'Val manifest:   {val_manifest_path}')

## 4. Depth/Color Pair Validation and File Integrity Checks

In [ ]:
def validate_h5_sample(h5_path, min_depth=0.001, max_depth=80.0):
    report = {'h5_path': h5_path, 'ok': False, 'reason': '', 'h': None, 'w': None, 'dmin': None, 'dmax': None}
    try:
        with h5py.File(h5_path, 'r') as f:
            if 'rgb_scene' not in f or 'depth_planar_m' not in f:
                report['reason'] = 'missing required datasets'
                return report

            rgb = f['rgb_scene'][()]
            depth = f['depth_planar_m'][()]

            if rgb.ndim != 3 or rgb.shape[2] != 3:
                report['reason'] = f'invalid rgb shape: {rgb.shape}'
                return report
            if depth.ndim != 2:
                report['reason'] = f'invalid depth shape: {depth.shape}'
                return report
            if rgb.shape[:2] != depth.shape[:2]:
                report['reason'] = f'shape mismatch rgb={rgb.shape} depth={depth.shape}'
                return report

            valid = np.isfinite(depth) & (depth > 0)
            if valid.sum() == 0:
                report['reason'] = 'no valid depth pixels'
                return report

            dmin = float(np.min(depth[valid]))
            dmax = float(np.max(depth[valid]))
            report.update({'h': int(depth.shape[0]), 'w': int(depth.shape[1]), 'dmin': dmin, 'dmax': dmax})

            if dmax < min_depth or dmin > max_depth:
                report['reason'] = 'depth out of expected range'
                return report

            report['ok'] = True
            return report
    except Exception as e:
        report['reason'] = str(e)
        return report

def validate_manifest(manifest_path, max_items=None):
    rows = []
    with open(manifest_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if max_items is not None and i >= max_items:
                break
            rows.append(row['h5_path'])

    reports = [validate_h5_sample(p, CONFIG['min_depth_m'], CONFIG['max_depth_m']) for p in tqdm(rows)]
    ok_reports = [r for r in reports if r['ok']]
    bad_reports = [r for r in reports if not r['ok']]
    return ok_reports, bad_reports

train_ok, train_bad = validate_manifest(train_manifest_path)
val_ok, val_bad = validate_manifest(val_manifest_path)

print(f'Train valid: {len(train_ok)} | Train invalid: {len(train_bad)}')
print(f'Val valid:   {len(val_ok)} | Val invalid:   {len(val_bad)}')
if train_bad[:5]:
    print('Sample broken train files:')
    for x in train_bad[:5]:
        print('-', x['h5_path'], '|', x['reason'])

## 5. Custom Dataset Class for Depth Anything V2

In [ ]:
class H5DepthDataset(Dataset):
    def __init__(self, samples, mode='train', transform=None, aug=None, config=None):
        self.samples = samples
        self.mode = mode
        self.transform = transform
        self.aug = aug
        self.config = config or {}

    def __len__(self):
        return len(self.samples)

    def _read_h5(self, path):
        with h5py.File(path, 'r') as f:
            rgb = f['rgb_scene'][()].astype(np.uint8)
            depth = f['depth_planar_m'][()].astype(np.float32)
            seg = f['segmentation_instance_id'][()].astype(np.uint32) if 'segmentation_instance_id' in f else None
        return rgb, depth, seg

    def __getitem__(self, idx):
        path = self.samples[idx]['h5_path']
        rgb, depth, seg = self._read_h5(path)

        if self.config.get('depth_clip_enabled', True):
            depth = np.clip(depth, self.config.get('min_depth_m', 0.001), self.config.get('max_depth_m', 80.0))

        if self.config.get('depth_normalize_enabled', False):
            depth = depth / max(self.config.get('max_depth_m', 80.0), 1e-6)

        if self.mode == 'train' and self.aug is not None:
            augmented = self.aug(image=rgb, mask=depth)
            rgb = augmented['image']
            depth = augmented['mask']

        sample = {'image': rgb / 255.0, 'depth': depth}
        if self.transform is not None:
            sample = self.transform(sample)

        image_t = torch.from_numpy(sample['image']).float()
        depth_t = torch.from_numpy(sample['depth']).float()

        out = {'image': image_t, 'depth': depth_t, 'path': path}
        if seg is not None:
            out['segmentation'] = torch.from_numpy(seg.astype(np.int64))
        return out

## 6. Transform and Augmentation Pipeline

In [ ]:
def build_transforms(mode, img_size=(518, 518)):
    net_w, net_h = img_size
    base = [
        Resize(
            width=net_w,
            height=net_h,
            resize_target=True if mode == 'train' else False,
            keep_aspect_ratio=True,
            ensure_multiple_of=14,
            resize_method='lower_bound',
            image_interpolation_method=cv2.INTER_CUBIC,
        ),
        NormalizeImage(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        PrepareForNet(),
    ]
    if mode == 'train':
        base = base + [Crop(img_size[0])]
    return Compose(base)

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(hue=0.1, contrast=0.1, brightness=0.1, saturation=0.1, p=0.5),
    A.GaussNoise(var_limit=(1.0, 25.0), p=0.3),
])

train_transform = build_transforms('train', CONFIG['img_size'])
val_transform = build_transforms('val', CONFIG['img_size'])
print('Transforms ready.')

## 7. DataLoader Factory with Train/Validation Settings

In [ ]:
def apply_segmentation_filter(samples, min_ratio=0.001):
    if not CONFIG.get('segmentation_filter_enabled', False):
        return samples
    kept = []
    for s in tqdm(samples, desc='Segmentation filter'):
        try:
            with h5py.File(s['h5_path'], 'r') as f:
                if 'segmentation_instance_id' not in f:
                    continue
                seg = f['segmentation_instance_id'][()]
                fg = np.mean(seg > 0)
                if fg >= min_ratio:
                    kept.append(s)
        except Exception:
            continue
    return kept

train_samples_filtered = apply_segmentation_filter(
    [{'h5_path': x['h5_path']} for x in train_ok],
    min_ratio=CONFIG['segmentation_min_foreground_ratio']
 )
val_samples_filtered = apply_segmentation_filter(
    [{'h5_path': x['h5_path']} for x in val_ok],
    min_ratio=CONFIG['segmentation_min_foreground_ratio']
 )

def get_dataloaders(config):
    train_ds = H5DepthDataset(
        samples=train_samples_filtered,
        mode='train',
        transform=train_transform,
        aug=train_aug,
        config=config
    )
    val_ds = H5DepthDataset(
        samples=val_samples_filtered,
        mode='val',
        transform=val_transform,
        aug=None,
        config=config
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=config['batch_size_train'],
        shuffle=True,
        num_workers=config['num_workers'],
        drop_last=True,
        pin_memory=config['pin_memory'],
        persistent_workers=config['persistent_workers'] if config['num_workers'] > 0 else False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=config['batch_size_val'],
        shuffle=False,
        num_workers=config['num_workers'],
        drop_last=False,
        pin_memory=config['pin_memory'],
        persistent_workers=config['persistent_workers'] if config['num_workers'] > 0 else False,
    )
    return train_loader, val_loader

train_loader, val_loader = get_dataloaders(CONFIG)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 8. Batch Visualization and Preprocessing Sanity Checks

In [ ]:
def denormalize_image(img_chw):
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    return np.clip(img_chw * std + mean, 0.0, 1.0)

def show_batch(loader, n=4, title='batch preview'):
    batch = next(iter(loader))
    imgs = batch['image'].cpu().numpy()
    depths = batch['depth'].cpu().numpy()

    n = min(n, imgs.shape[0])
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(n):
        rgb = denormalize_image(imgs[i])
        rgb = np.transpose(rgb, (1, 2, 0))
        d = depths[i]

        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title('RGB')
        axes[i, 0].axis('off')

        im = axes[i, 1].imshow(d, cmap='viridis')
        axes[i, 1].set_title('Depth')
        axes[i, 1].axis('off')
        fig.colorbar(im, ax=axes[i, 1])

    plt.suptitle(title)
    plt.tight_layout()

show_batch(train_loader, n=3, title='Train batch sanity check')
show_batch(val_loader, n=3, title='Val batch sanity check')

## 9. Metric Functions for Depth Evaluation

In [ ]:
def eval_depth(pred, target, eps=1e-6):
    pred = torch.clamp(pred, min=eps)
    target = torch.clamp(target, min=eps)
    assert pred.shape == target.shape

    thresh = torch.max(target / pred, pred / target)
    d1 = torch.mean((thresh < 1.25).float())

    diff = pred - target
    diff_log = torch.log(pred) - torch.log(target)

    abs_rel = torch.mean(torch.abs(diff) / target)
    rmse = torch.sqrt(torch.mean(diff ** 2))
    mae = torch.mean(torch.abs(diff))
    silog = torch.sqrt(torch.mean(diff_log ** 2) - 0.5 * (torch.mean(diff_log) ** 2))

    return {
        'd1': d1.detach(),
        'abs_rel': abs_rel.detach(),
        'rmse': rmse.detach(),
        'mae': mae.detach(),
        'silog': silog.detach(),
    }

## 10. Model Initialization with Pretrained Encoder Weights

In [ ]:
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]},
}

def init_model_and_optim(config, train_loader_len):
    model = DepthAnythingV2(**{
        **model_configs[config['model_encoder']],
        'max_depth': config['max_depth_m'],
    })

    ckpt = torch.load(config['pretrained_weights_path'], map_location='cpu')
    model.load_state_dict({k: v for k, v in ckpt.items() if 'pretrained' in k}, strict=False)

    optim = torch.optim.AdamW([
        {'params': [p for n, p in model.named_parameters() if 'pretrained' in n], 'lr': config['learning_rate']},
        {'params': [p for n, p in model.named_parameters() if 'pretrained' not in n], 'lr': config['learning_rate'] * 10.0},
    ], lr=config['learning_rate'], weight_decay=config['weight_decay'])

    criterion = SiLogLoss()
    warmup_steps = int(train_loader_len * config['warmup_epochs'])
    total_steps = int(config['num_epochs'] * config['scheduler_rate'] * train_loader_len)
    scheduler = transformers.get_cosine_schedule_with_warmup(
        optim,
        num_warmup_steps=warmup_steps,
        num_training_steps=max(total_steps, warmup_steps + 1),
    )
    return model, optim, criterion, scheduler

print('Model config ready.')

## 11. Training Loop with accelerate and Mixed Precision

In [ ]:
def train_one_epoch(model, loader, criterion, optim, scheduler, accelerator, config):
    model.train()
    running = 0.0

    for batch in tqdm(loader, disable=not accelerator.is_local_main_process, desc='train'):
        optim.zero_grad()
        img = batch['image']
        depth = batch['depth']

        pred = model(img)
        valid_mask = (depth <= config['max_depth_m']) & (depth >= config['min_depth_m'])
        loss = criterion(pred, depth, valid_mask)

        accelerator.backward(loss)
        optim.step()
        scheduler.step()
        running += loss.detach()

    running = running / max(len(loader), 1)
    return accelerator.reduce(running, reduction='mean').item()

## 12. Validation Loop and Checkpointing Best Model

In [ ]:
def validate(model, loader, accelerator, config):
    model.eval()
    results = {'d1': 0, 'abs_rel': 0, 'rmse': 0, 'mae': 0, 'silog': 0}

    for batch in tqdm(loader, disable=not accelerator.is_local_main_process, desc='val'):
        img = batch['image'].float()
        depth = batch['depth'][0]

        with torch.no_grad():
            pred = model(img)
            pred = F.interpolate(
                pred[:, None],
                size=depth.shape[-2:],
                mode='bilinear',
                align_corners=True
            )[0, 0]

        valid_mask = (depth <= config['max_depth_m']) & (depth >= config['min_depth_m'])
        m = eval_depth(pred[valid_mask], depth[valid_mask])
        for k in results:
            results[k] += m[k]

    for k in results:
        results[k] = results[k] / max(len(loader), 1)
        results[k] = round(accelerator.reduce(results[k], reduction='mean').item(), 4)
    return results

def train_fn(config):
    set_seed(config['seed'])
    ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)
    accelerator = Accelerator(
        mixed_precision=config['mixed_precision'],
        kwargs_handlers=[ddp_kwargs],
    )

    train_loader, val_loader = get_dataloaders(config)
    model, optim, criterion, scheduler = init_model_and_optim(config, len(train_loader))

    model, optim, train_loader, val_loader, scheduler = accelerator.prepare(
        model, optim, train_loader, val_loader, scheduler
    )

    best_abs_rel = float('inf')
    ckpt_root = Path(config['checkpoint_root'])
    ckpt_root.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, config['num_epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optim, scheduler, accelerator, config)
        val_metrics = validate(model, val_loader, accelerator, config)

        accelerator.wait_for_everyone()
        accelerator.save_state(str(ckpt_root / 'latest_state'), safe_serialization=False)

        if val_metrics['abs_rel'] < best_abs_rel:
            best_abs_rel = val_metrics['abs_rel']
            unwrapped = accelerator.unwrap_model(model)
            if accelerator.is_local_main_process:
                torch.save(unwrapped.state_dict(), ckpt_root / 'best_model.pt')

        accelerator.print(f"Epoch {epoch}: train_loss={train_loss:.5f}, val={val_metrics}")

    return {'best_abs_rel': best_abs_rel}

## 13. Inference on Validation Samples and Depth Map Rendering

In [ ]:
def run_inference_preview(config, num_images=5, device='cuda'):
    model = DepthAnythingV2(**{
        **model_configs[config['model_encoder']],
        'max_depth': config['max_depth_m'],
    }).to(device)

    best_ckpt = Path(config['checkpoint_root']) / 'best_model.pt'
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Best checkpoint not found: {best_ckpt}')

    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    model.eval()

    _, val_loader_local = get_dataloaders(config)
    fig, axes = plt.subplots(num_images, 3, figsize=(15, 4 * num_images))
    if num_images == 1:
        axes = np.expand_dims(axes, axis=0)

    with torch.no_grad():
        for i, batch in enumerate(val_loader_local):
            if i >= num_images:
                break
            img_t = batch['image'].to(device)
            depth_gt = batch['depth'][0].cpu()

            pred = model(img_t)
            pred = F.interpolate(
                pred[:, None],
                size=depth_gt.shape[-2:],
                mode='bilinear',
                align_corners=True
            )[0, 0].cpu()

            rgb = denormalize_image(batch['image'][0].cpu().numpy())
            rgb = np.transpose(rgb, (1, 2, 0))
            vmax = max(float(depth_gt.max()), float(pred.max()))

            axes[i, 0].imshow(rgb)
            axes[i, 0].set_title('RGB')
            axes[i, 0].axis('off')

            im1 = axes[i, 1].imshow(depth_gt, cmap='viridis', vmin=0, vmax=vmax)
            axes[i, 1].set_title('GT Depth')
            axes[i, 1].axis('off')
            fig.colorbar(im1, ax=axes[i, 1])

            im2 = axes[i, 2].imshow(pred, cmap='viridis', vmin=0, vmax=vmax)
            axes[i, 2].set_title('Pred Depth')
            axes[i, 2].axis('off')
            fig.colorbar(im2, ax=axes[i, 2])

    plt.tight_layout()

# Example after training:
# run_inference_preview(CONFIG, num_images=5, device='cuda')

## 14. Optional Dataset Caching/Serialization for Faster Reloads

In [ ]:
def export_cache_manifest(samples, out_csv, version='v1'):
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with open(out_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['h5_path', 'cache_version'])
        writer.writeheader()
        for s in samples:
            writer.writerow({'h5_path': s['h5_path'], 'cache_version': version})

cache_manifest_path = Path(CONFIG['processed_root']) / CONFIG['cache_manifest']
export_cache_manifest(
    train_samples_filtered + val_samples_filtered,
    cache_manifest_path,
    version='depth-anything-v2-h5-schema-rgb_scene-depth_planar_m-v1'
)
print(f'Cache manifest written: {cache_manifest_path}')

## End-to-End Execution Cell

In [ ]:
# Run training end-to-end (single process).
# For multi-GPU notebook launch, adapt this to notebook_launcher(train_fn, num_processes=N).

result = train_fn(CONFIG)
print('Training finished:', result)

# Optional post-training preview
# run_inference_preview(CONFIG, num_images=5, device='cuda')